In [3]:
# Data handling and mathematics
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Feature preprocessing
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Feature selection
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression

# Principal Component Analysis
from sklearn.decomposition import PCA

# Time-series validation and feature evaluation
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# File and folder paths
from pathlib import Path

# Plot settings
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print("Feature Engineering and PCA libraries imported successfully.")

Feature Engineering and PCA libraries imported successfully.


In [6]:
# Load the completed Data Mining dataset
df_fe = pd.read_csv("../data/energy_1hour_data_mining.csv")

print("Dataset shape:", df_fe.shape)

print("\nAll columns and data types:")
for i, (column, dtype) in enumerate(df_fe.dtypes.items(), start=1):
    print(f"{i:3}. {column:45} {dtype}")

print("\nTotal missing values:", df_fe.isna().sum().sum())
print("Duplicate rows:", df_fe.duplicated().sum())

Dataset shape: (3289, 105)

All columns and data types:
  1. date                                          str
  2. Appliances                                    int64
  3. lights                                        int64
  4. Total_Energy                                  int64
  5. T1                                            float64
  6. T2                                            float64
  7. T3                                            float64
  8. T4                                            float64
  9. T5                                            float64
 10. T6                                            float64
 11. T7                                            float64
 12. T8                                            float64
 13. T9                                            float64
 14. RH_1                                          float64
 15. RH_2                                          float64
 16. RH_3                                          float64
 17. RH_4 

In [11]:
# Select columns 31–82 (Python positions 30–81)
outlier_columns = df_fe.columns[25:77].tolist()

# Verify the expected boundaries before removing columns
#assert outlier_columns[0] == "Energy_IQR_Outlier", "Column order differs—stop and review."
#assert outlier_columns[-1] == "spike_outlier", "Column order differs—stop and review."

df_fe = df_fe.drop(columns=outlier_columns)

print("Removed columns:", len(outlier_columns))
print("Updated dataset shape:", df_fe.shape)

print("\nRemaining columns:")
for i, column in enumerate(df_fe.columns, start=1):
    print(f"{i:3}. {column}")

Removed columns: 52
Updated dataset shape: (3289, 48)

Remaining columns:
  1. date
  2. Appliances
  3. lights
  4. Total_Energy
  5. T1
  6. T2
  7. T3
  8. T4
  9. T5
 10. T6
 11. T7
 12. T8
 13. T9
 14. RH_1
 15. RH_2
 16. RH_3
 17. RH_4
 18. RH_5
 19. RH_6
 20. RH_7
 21. RH_8
 22. RH_9
 23. T_out
 24. Press_mm_hg
 25. RH_out
 26. records_per_hour
 27. hour
 28. day_of_week
 29. day_name
 30. is_weekend
 31. day
 32. Is_Outlier
 33. High_Outlier
 34. Hour
 35. Temp_Group
 36. Floor_1_Temp
 37. Floor_2_Temp
 38. Floor_1_Temp_Diff
 39. Floor_2_Temp_Diff
 40. Floor_1_Humidity
 41. Floor_2_Humidity
 42. Outlier_Status
 43. H1_Likely_Genuine_Candidate
 44. Floor_1_NextHour_Change
 45. Floor_2_NextHour_Change
 46. Both_Floors_Warming
 47. Temperature_Supported_Candidate
 48. Evening_Genuine_Candidate


In [19]:
df_fe["day"]

0       2016-01-11
1       2016-01-11
2       2016-01-11
3       2016-01-11
4       2016-01-11
           ...    
3284    2016-05-27
3285    2016-05-27
3286    2016-05-27
3287    2016-05-27
3288    2016-05-27
Name: day, Length: 3289, dtype: str

In [20]:
df_fe["day_name"]

0       Monday
1       Monday
2       Monday
3       Monday
4       Monday
         ...  
3284    Friday
3285    Friday
3286    Friday
3287    Friday
3288    Friday
Name: day_name, Length: 3289, dtype: str

In [21]:
# Read original column names without loading the full dataset
original_columns = pd.read_csv(
    "../data/energy_1hour_data_mining.csv",
    nrows=0
).columns.tolist()

removed_columns = [
    column for column in original_columns
    if column not in df_fe.columns
]

print("Original column count:", len(original_columns))
print("Current column count:", df_fe.shape[1])

print("\nColumns removed from df_fe:")
for column in removed_columns:
    print(column)

Original column count: 105
Current column count: 48

Columns removed from df_fe:
Windspeed
Visibility
Tdewpoint
rv1
rv2
Energy_IQR_Outlier
High_Appliances
High_Energy
High_T_out
High_Windspeed
Outlier_Flag
Press_mm_hg_IQR_flag
Press_outlier
RH5_High
RH7_outlier
RH8_outlier_high
RH9_outlier
RH9_outlier_high
RH9_outlier_low
RH_1_IQR_flag
RH_2_IQR_flag
RH_3_IQR_flag
RH_4_IQR_flag
RH_5_IQR_flag
RH_6_IQR_flag
RH_7_IQR_flag
RH_8_IQR_flag
RH_9_IQR_flag
RH_out_IQR_flag
RH_out_outlier
T1_IQR_flag
T2_IQR_flag
T2_outlier
T3_IQR_flag
T3_outlier
T4_IQR_flag
T5_IQR_flag
T5_outlier
T6_IQR_flag
T6_outlier
T7_IQR_flag
T7_outlier
T8_IQR_flag
T8_outlier
T8_outlier_low
T9_IQR_flag
T_out_IQR_flag
T_out_outlier
Tdewpoint_IQR_flag
Total_Energy_Outlier
Visibility_IQR_flag
Weekend_High_Energy
Windspeed_IQR_flag
Windspeed_outlier
rv1_IQR_flag
rv2_IQR_flag
spike_outlier


In [22]:
# Reload the original dataset
df_original = pd.read_csv("../data/energy_1hour_data_mining.csv")

# Explicitly select columns for Feature Engineering
keep_columns = [
    "date", "Total_Energy",
    "T1", "T2", "T3", "T4", "T5", "T6", "T7", "T8", "T9",
    "RH_1", "RH_2", "RH_3", "RH_4", "RH_5",
    "RH_6", "RH_7", "RH_8", "RH_9",
    "T_out", "Press_mm_hg", "RH_out",
    "Windspeed", "Visibility", "Tdewpoint",
    "records_per_hour",
    "hour", "day_of_week", "is_weekend", "day",
    "Floor_1_Temp", "Floor_2_Temp",
    "Floor_1_Temp_Diff", "Floor_2_Temp_Diff",
    "Floor_1_Humidity", "Floor_2_Humidity"
]

df_fe = df_original.loc[:, keep_columns].copy()

print("Feature Engineering dataset shape:", df_fe.shape)
print("\nRestored weather measurements:")
display(df_fe[["Windspeed", "Visibility", "Tdewpoint"]].head())

print("\nMissing values:", df_fe.isna().sum().sum())

Feature Engineering dataset shape: (3289, 37)

Restored weather measurements:


,Windspeed,Visibility,Tdewpoint
0,6.166667,53.416667,5.050000
1,5.416667,40.000000,4.658333
2,6.000000,40.000000,4.391667
3,6.000000,40.000000,4.016667
4,6.000000,40.000000,3.816667



Missing values: 0


In [23]:
# Parse timestamps and ensure chronological order
df_fe["date"] = pd.to_datetime(df_fe["date"], errors="raise")
df_fe = df_fe.sort_values("date").reset_index(drop=True)

# Derive hour directly from the timestamp
df_fe["hour"] = df_fe["date"].dt.hour

# Create cyclical hour features
df_fe["hour_sin"] = np.sin(2 * np.pi * df_fe["hour"] / 24)
df_fe["hour_cos"] = np.cos(2 * np.pi * df_fe["hour"] / 24)

# Show one example for every available hour
display(
    df_fe[["hour", "hour_sin", "hour_cos"]]
    .drop_duplicates("hour")
    .sort_values("hour")
    .round(3)
    .reset_index(drop=True)
)

print("Updated shape:", df_fe.shape)

,hour,hour_sin,hour_cos
0,0,0.000,1.000
1,1,0.259,0.966
2,2,0.500,0.866
3,3,0.707,0.707
4,4,0.866,0.500
5,5,0.966,0.259
6,6,1.000,0.000
7,7,0.966,-0.259
8,8,0.866,-0.500
9,9,0.707,-0.707


Updated shape: (3289, 39)


In [24]:
# Create the proposed afternoon peak-window feature
df_fe["is_afternoon_peak"] = (
    (df_fe["hour"] >= 15) & (df_fe["hour"] < 19)
).astype(int)

# Verify the flag for each hour
display(
    df_fe[["hour", "is_afternoon_peak"]]
    .drop_duplicates("hour")
    .sort_values("hour")
    .reset_index(drop=True)
)

print("Updated shape:", df_fe.shape)

,hour,is_afternoon_peak
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0
5,5,0
6,6,0
7,7,0
8,8,0
9,9,0


Updated shape: (3289, 40)


In [25]:
# Extract month from the timestamp
df_fe["month"] = df_fe["date"].dt.month

# Encode January–May, using January as the reference month
month_category = pd.Categorical(
    df_fe["month"],
    categories=[1, 2, 3, 4, 5],
    ordered=False
)

month_features = pd.get_dummies(
    month_category,
    prefix="month",
    drop_first=True,
    dtype=int
)
month_features.index = df_fe.index

# Assign columns safely, including when rerunning this cell
df_fe[month_features.columns] = month_features

# Verify one example per month
display(
    df_fe[
        ["month", "month_2", "month_3", "month_4", "month_5"]
    ]
    .drop_duplicates("month")
    .sort_values("month")
    .reset_index(drop=True)
)

print("Updated shape:", df_fe.shape)

,month,month_2,month_3,month_4,month_5
0,1,0,0,0,0
1,2,1,0,0,0
2,3,0,1,0,0
3,4,0,0,1,0
4,5,0,0,0,1


Updated shape: (3289, 45)


In [26]:
# Identify rows with a valid previous-hour observation
previous_hour_available = (
    df_fe["date"].diff() == pd.Timedelta(hours=1)
)

# Current floor temperature minus previous-hour floor temperature
df_fe["Floor_1_Temp_Change"] = (
    df_fe["Floor_1_Temp"].diff().where(previous_hour_available)
)

df_fe["Floor_2_Temp_Change"] = (
    df_fe["Floor_2_Temp"].diff().where(previous_hour_available)
)

# Keep unavailable changes as missing rather than labeling them "not warming"
df_fe["Both_Floors_Warming"] = (
    (df_fe["Floor_1_Temp_Change"] > 0)
    & (df_fe["Floor_2_Temp_Change"] > 0)
).astype("Int64").where(previous_hour_available)

display(
    df_fe[
        ["date", "Floor_1_Temp_Change",
         "Floor_2_Temp_Change", "Both_Floors_Warming"]
    ].head(10)
)

print("Warming flag counts:")
print(df_fe["Both_Floors_Warming"].value_counts(dropna=False))
print("\nUpdated shape:", df_fe.shape)

,date,Floor_1_Temp_Change,Floor_2_Temp_Change,Both_Floors_Warming
0,2016-01-11 17:00:00,NaN,NaN,<NA>
1,2016-01-11 18:00:00,0.018889,-0.023750,0
2,2016-01-11 19:00:00,0.512222,0.137750,1
3,2016-01-11 20:00:00,0.317593,0.217611,1
4,2016-01-11 21:00:00,0.282222,0.331556,1
5,2016-01-11 22:00:00,0.224815,0.287306,1
6,2016-01-11 23:00:00,-0.054398,0.192583,0
7,2016-01-12 00:00:00,-0.245417,-0.089500,0
8,2016-01-12 01:00:00,-0.205833,0.144667,0
9,2016-01-12 02:00:00,-0.165370,-0.092972,0


Warming flag counts:
Both_Floors_Warming
0       2451
1        837
<NA>       1
Name: count, dtype: Int64

Updated shape: (3289, 48)


In [27]:
# Fill missing values only in the new temperature-change features
warming_columns = [
    "Floor_1_Temp_Change",
    "Floor_2_Temp_Change",
    "Both_Floors_Warming"
]

df_fe[warming_columns] = df_fe[warming_columns].fillna(0)

display(df_fe[["date"] + warming_columns].head())

print("Remaining missing values:")
print(df_fe[warming_columns].isna().sum())

,date,Floor_1_Temp_Change,Floor_2_Temp_Change,Both_Floors_Warming
0,2016-01-11 17:00:00,0.000000,0.000000,0
1,2016-01-11 18:00:00,0.018889,-0.023750,0
2,2016-01-11 19:00:00,0.512222,0.137750,1
3,2016-01-11 20:00:00,0.317593,0.217611,1
4,2016-01-11 21:00:00,0.282222,0.331556,1


Remaining missing values:
Floor_1_Temp_Change    0
Floor_2_Temp_Change    0
Both_Floors_Warming    0
dtype: int64


In [28]:
# Combine the existing weekend and afternoon indicators
df_fe["Weekend_Afternoon"] = (
    df_fe["is_weekend"].astype(int)
    * df_fe["is_afternoon_peak"]
)

# Verify the four possible combinations
display(
    df_fe[
        ["is_weekend", "is_afternoon_peak", "Weekend_Afternoon"]
    ]
    .drop_duplicates()
    .sort_values(["is_weekend", "is_afternoon_peak"])
    .reset_index(drop=True)
)

print("Updated shape:", df_fe.shape)

,is_weekend,is_afternoon_peak,Weekend_Afternoon
0,False,0,0
1,False,1,0
2,True,0,0
3,True,1,1


Updated shape: (3289, 49)


In [29]:
# Check how many 10-minute measurements contributed to each hour
print("Records per hour — frequency:")
print(df_fe["records_per_hour"].value_counts().sort_index())

# Inspect hours that do not contain exactly six measurements
incomplete_hours = df_fe.loc[
    df_fe["records_per_hour"] != 6,
    ["date", "records_per_hour", "Total_Energy"]
]

print("\nHours with a record count other than 6:", len(incomplete_hours))
display(incomplete_hours)

Records per hour — frequency:
records_per_hour
6    3289
Name: count, dtype: int64

Hours with a record count other than 6: 0


,date,records_per_hour,Total_Energy


In [30]:
# Remove the constant completeness-check column
df_fe = df_fe.drop(columns=["records_per_hour"])

print("Updated shape:", df_fe.shape)
print("Total missing values:", df_fe.isna().sum().sum())

Updated shape: (3289, 48)
Total missing values: 0


In [31]:
# Preserve timestamps for chronological splitting
timestamps = df_fe["date"].copy()

# Separate the prediction target
y = df_fe["Total_Energy"].copy()

# Select predictors using the engineered time representations
X = df_fe.drop(
    columns=["date", "Total_Energy", "hour", "month"]
).copy()

print("Input shape X:", X.shape)
print("Target shape y:", y.shape)

print("\nPredictor columns:")
for i, column in enumerate(X.columns, start=1):
    print(f"{i:2}. {column}")

print("\nNon-numeric predictor columns:")
print(X.select_dtypes(exclude=["number", "bool"]).columns.tolist())

Input shape X: (3289, 44)
Target shape y: (3289,)

Predictor columns:
 1. T1
 2. T2
 3. T3
 4. T4
 5. T5
 6. T6
 7. T7
 8. T8
 9. T9
10. RH_1
11. RH_2
12. RH_3
13. RH_4
14. RH_5
15. RH_6
16. RH_7
17. RH_8
18. RH_9
19. T_out
20. Press_mm_hg
21. RH_out
22. Windspeed
23. Visibility
24. Tdewpoint
25. day_of_week
26. is_weekend
27. day
28. Floor_1_Temp
29. Floor_2_Temp
30. Floor_1_Temp_Diff
31. Floor_2_Temp_Diff
32. Floor_1_Humidity
33. Floor_2_Humidity
34. hour_sin
35. hour_cos
36. is_afternoon_peak
37. month_2
38. month_3
39. month_4
40. month_5
41. Floor_1_Temp_Change
42. Floor_2_Temp_Change
43. Both_Floors_Warming
44. Weekend_Afternoon

Non-numeric predictor columns:
['day']


In [32]:
# Remove redundant temperature differences from model inputs
X = X.drop(
    columns=["Floor_1_Temp_Diff", "Floor_2_Temp_Diff"],
    errors="ignore"
)

print("Updated input shape:", X.shape)

Updated input shape: (3289, 42)


In [33]:
# Inspect the meaning of the non-numeric "day" column
print("Data type:", X["day"].dtype)
print("Unique values:", X["day"].nunique())

display(
    df_fe[["date", "day", "day_of_week", "is_weekend"]].head(10)
)

print("\nMost frequent values:")
print(X["day"].value_counts().head(10))

Data type: str
Unique values: 138


,date,day,day_of_week,is_weekend
0,2016-01-11 17:00:00,2016-01-11,0,False
1,2016-01-11 18:00:00,2016-01-11,0,False
2,2016-01-11 19:00:00,2016-01-11,0,False
3,2016-01-11 20:00:00,2016-01-11,0,False
4,2016-01-11 21:00:00,2016-01-11,0,False
5,2016-01-11 22:00:00,2016-01-11,0,False
6,2016-01-11 23:00:00,2016-01-11,0,False
7,2016-01-12 00:00:00,2016-01-12,1,False
8,2016-01-12 01:00:00,2016-01-12,1,False
9,2016-01-12 02:00:00,2016-01-12,1,False



Most frequent values:
day
2016-01-12    24
2016-01-13    24
2016-01-14    24
2016-01-15    24
2016-01-16    24
2016-01-17    24
2016-01-18    24
2016-01-19    24
2016-01-20    24
2016-01-21    24
Name: count, dtype: int64


In [34]:
# Remove the date-text column from predictors
X = X.drop(columns=["day"], errors="ignore")

print("Updated input shape:", X.shape)

print("\nNon-numeric predictor columns:")
print(X.select_dtypes(exclude=["number", "bool"]).columns.tolist())

Updated input shape: (3289, 41)

Non-numeric predictor columns:
[]


### Chronological Train–Test Split

The 3,289 hourly observations were divided chronologically. The earliest 80% were assigned to training and the latest 20% to testing.

| Dataset | Observations | Predictors | Start | End |
|---|---:|---:|---|---|
| Training | 2,631 | 41 | 2016-01-11 17:00 | 2016-04-30 07:00 |
| Testing | 658 | 41 | 2016-04-30 08:00 | 2016-05-27 17:00 |

**Target:** `Total_Energy` for the current hour.

This split evaluates the model on later observations while preserving temporal order. Scaling and PCA will be fitted using training data only, then applied to the test data.

**Month-feature limitation:** May appears only in the test period. Therefore, `month_5` is always zero during training, and the model cannot learn a May-specific effect. We will address this when finalizing the predictors.

In [35]:
# Verify chronological order before splitting
assert timestamps.is_monotonic_increasing, "Timestamps must be sorted."

# Split the earliest 80% and latest 20%
split_index = int(len(X) * 0.80)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

print("Training inputs:", X_train.shape)
print("Testing inputs:", X_test.shape)

print("\nTraining period:")
print(timestamps.iloc[0], "to", timestamps.iloc[split_index - 1])

print("\nTesting period:")
print(timestamps.iloc[split_index], "to", timestamps.iloc[-1])

Training inputs: (2631, 41)
Testing inputs: (658, 41)

Training period:
2016-01-11 17:00:00 to 2016-04-30 07:00:00

Testing period:
2016-04-30 08:00:00 to 2016-05-27 17:00:00


In [36]:
# Check that inputs, target, and timestamps have matching rows
assert X.index.equals(y.index)
assert X.index.equals(timestamps.index)

# Combine everything into one dataset
pca_ready = X.copy()
pca_ready.insert(0, "date", timestamps)
pca_ready["Total_Energy"] = y

# Save in the project's data folder
save_path = Path("../data/energy_1hour_fe_ready.csv")
save_path.parent.mkdir(parents=True, exist_ok=True)
pca_ready.to_csv(save_path, index=False)

print("Saved:", save_path)
print("Dataset shape:", pca_ready.shape)

Saved: ..\data\energy_1hour_fe_ready.csv
Dataset shape: (3289, 43)


In [4]:
# Load the saved Feature Engineering dataset
six_fe_ready = pd.read_csv(
    "../data/energy_1hour_fe_ready.csv",
    parse_dates=["date"]
).sort_values("date").reset_index(drop=True)

# Separate timestamps, target, and predictors
timestamps = six_fe_ready["date"].copy()
y = six_fe_ready["Total_Energy"].copy()
X = six_fe_ready.drop(columns=["date", "Total_Energy"]).copy()

# Restore the chronological 80%–20% split
split_index = int(len(X) * 0.80)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()
y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

# Select your six features for PCA
floor_weather_columns = [
    "Floor_1_Temp",
    "Floor_2_Temp",
    "Floor_1_Humidity",
    "Floor_2_Humidity",
    "T_out",
    "RH_out"
]

X_train_floor_input = X_train[floor_weather_columns].copy()
X_test_floor_input = X_test[floor_weather_columns].copy()

print("Training PCA inputs:", X_train_floor_input.shape)
print("Testing PCA inputs:", X_test_floor_input.shape)

display(X_train_floor_input.head())

Training PCA inputs: (2631, 6)
Testing PCA inputs: (658, 6)


,Floor_1_Temp,Floor_2_Temp,Floor_1_Humidity,Floor_2_Humidity,T_out,RH_out
0,19.626667,17.688778,45.342361,47.282528,6.308333,92.000000
1,19.645556,17.665028,45.060417,47.395694,5.941667,91.583333
2,20.157778,17.802778,48.698194,48.645583,6.000000,89.750000
3,20.475370,18.020389,47.118287,52.298028,6.000000,87.583333
4,20.757593,18.351944,45.631296,50.985000,5.833333,87.416667


In [5]:
# Learn scaling parameters from the six training features only
six_scaler = StandardScaler()

Z_train_six = pd.DataFrame(
    six_scaler.fit_transform(X_train_floor_input),
    columns=floor_weather_columns,
    index=X_train_floor_input.index
)

# Apply the training parameters to the test features
Z_test_six = pd.DataFrame(
    six_scaler.transform(X_test_floor_input),
    columns=floor_weather_columns,
    index=X_test_floor_input.index
)

# Check training standardization
display(
    pd.DataFrame({
        "Training_mean": Z_train_six.mean(),
        "Training_std": Z_train_six.std(ddof=0)
    }).round(6)
)

,Training_mean,Training_std
Floor_1_Temp,0.0,1.0
Floor_2_Temp,-0.0,1.0
Floor_1_Humidity,0.0,1.0
Floor_2_Humidity,-0.0,1.0
T_out,-0.0,1.0
RH_out,-0.0,1.0


In [6]:
# Fit PCA on the six standardized training features only
six_pca = PCA(svd_solver="full")
six_pca.fit(Z_train_six)

component_names = [
    f"PC{i}" for i in range(1, six_pca.n_components_ + 1)
]

# Feature weights defining each component
six_pca_weights = pd.DataFrame(
    six_pca.components_.T,
    index=floor_weather_columns,
    columns=component_names
)

print("PCA weights:")
display(six_pca_weights.round(4))

# Individual and cumulative explained variance
six_variance_table = pd.DataFrame({
    "Component": component_names,
    "Eigenvalue": six_pca.explained_variance_,
    "Explained_variance_%": six_pca.explained_variance_ratio_ * 100,
    "Cumulative_variance_%": (
        np.cumsum(six_pca.explained_variance_ratio_) * 100
    )
})

print("\nExplained variance:")
display(six_variance_table.round(4))

PCA weights:


,PC1,PC2,PC3,PC4,PC5,PC6
Floor_1_Temp,0.4619,0.3493,0.3590,-0.3662,-0.3755,-0.5106
Floor_2_Temp,0.5133,0.1155,0.5155,0.1156,0.4478,0.4935
Floor_1_Humidity,-0.2638,0.5973,-0.0633,0.0717,0.6387,-0.3957
Floor_2_Humidity,-0.3170,0.5591,-0.0378,-0.4407,-0.2538,0.5718
T_out,0.3696,0.4312,-0.4363,0.6213,-0.3022,0.0992
RH_out,-0.4654,0.0971,0.6400,0.5169,-0.3080,-0.0488



Explained variance:


,Component,Eigenvalue,Explained_variance_%,Cumulative_variance_%
0,PC1,2.6037,43.3793,43.3793
1,PC2,2.0351,33.9052,77.2845
2,PC3,0.8212,13.6812,90.9657
3,PC4,0.2426,4.0413,95.0070
4,PC5,0.1658,2.7628,97.7698
5,PC6,0.1339,2.2302,100.0000


In [8]:
# Version 1: 24 original measurements -> 11 components
room_columns = (
    [f"T{i}" for i in range(1, 10)]
    + [f"RH_{i}" for i in range(1, 10)]
    + [
        "T_out", "Press_mm_hg", "RH_out",
        "Windspeed", "Visibility", "Tdewpoint"
    ]
)

room_scaler = StandardScaler()

Z_train_room = room_scaler.fit_transform(X_train[room_columns])
Z_test_room = room_scaler.transform(X_test[room_columns])

pca_11 = PCA(n_components=11, svd_solver="full")
pc11_names = [f"PC{i}" for i in range(1, 12)]

X_train_pc11 = pd.DataFrame(
    pca_11.fit_transform(Z_train_room),
    index=X_train.index,
    columns=pc11_names
)

X_test_pc11 = pd.DataFrame(
    pca_11.transform(Z_test_room),
    index=X_test.index,
    columns=pc11_names
)

# Version 2: six floor/weather measurements -> six components
floor_weather_columns = [
    "Floor_1_Temp",
    "Floor_2_Temp",
    "Floor_1_Humidity",
    "Floor_2_Humidity",
    "T_out",
    "RH_out"
]

six_scaler = StandardScaler()

Z_train_six = six_scaler.fit_transform(X_train[floor_weather_columns])
Z_test_six = six_scaler.transform(X_test[floor_weather_columns])

six_pca = PCA(n_components=6, svd_solver="full")
pc6_names = [f"PC{i}" for i in range(1, 7)]

X_train_pc6 = pd.DataFrame(
    six_pca.fit_transform(Z_train_six),
    index=X_train.index,
    columns=pc6_names
)

X_test_pc6 = pd.DataFrame(
    six_pca.transform(Z_test_six),
    index=X_test.index,
    columns=pc6_names
)

# Check the two representations
print("11-PC training/testing:", X_train_pc11.shape, X_test_pc11.shape)
print("6-PC training/testing:", X_train_pc6.shape, X_test_pc6.shape)

print(
    "\n11-PC retained training variance:",
    f"{pca_11.explained_variance_ratio_.sum():.4%}"
)
print(
    "6-PC retained training variance:",
    f"{six_pca.explained_variance_ratio_.sum():.4%}"
)

11-PC training/testing: (2631, 11) (658, 11)
6-PC training/testing: (2631, 6) (658, 6)

11-PC retained training variance: 95.2168%
6-PC retained training variance: 100.0000%
